# 03 진단 모델 학습

정답을 다시 만들고 7문항 로지스틱을 학습해, 계수를 보기별 점수표로 바꿔 저장한다.

- 근거: 데이터 처리 기획서 3-9 (정답식), 3-10 (처리 순서) / 신용진단_학습데이터_조사 8~10 / 02 탐색 노트북 보고서
- 2026-11-13 결정: 점수 구간은 학습 데이터 분위수로 경계 4개를 NICE 분포에 맞추고, 경계가 600/700/800/900점에 오도록 점수 축을 구간마다 맞춤 / 상한은 데이터 기준 (최대 800점, 700점), FastAPI에서 적용 / '잘 모르겠어요' = 가운데 보기 (짝수면 가운데 두 보기 평균), 상한은 실제로 고른 답에만
- 저장: 점수표 `ml/app/artifacts/credit_score_table.json` (GitHub에 올림, 데이터 행 없음) / 학습 기록 `ml/data/model/credit_train_log.json` (안 올림)
- 점수 계산은 `ml/app/credit.py`의 `score_answers` (04 평가, /credit/score와 같은 함수)
- GitHub에 올리기 전에 **출력 지우기**
- 맨 아래 **7. 학습 결과 보고서**에 결과와 근거를 정리

In [ ]:
import json
import sys
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

# ml 폴더 찾기 (노트북 위치: ml/data/ipynb)
ML_DIR = next(d for d in [Path.cwd(), *Path.cwd().parents] if (d / "app" / "clean.py").exists())
sys.path.insert(0, str(ML_DIR))

from app.credit import OPTIONS, SCORE_TABLE_PATH, SOURCE_COLUMNS, load_score_table, middle_points, score_answers, to_answers

CSV_DIR = ML_DIR / "data" / "csv"
TRAIN_LOG_PATH = ML_DIR / "data" / "model" / "credit_train_log.json"
LABEL = "결과값(연체회차)"

SEED = 42               # 난수 고정 (정답 뽑기, 학습/평가 나누기)
NOISE_SD = 0.5          # 정답식 잡음 표준편차 (기획서 3-9)
SIGNAL_QUESTIONS = ["Q1", "Q2", "Q3", "Q7", "Q8"]                 # 학원 신호 (기획서 3-9)
SCORE_QUESTIONS = ["Q1", "Q2", "Q3", "Q7", "Q8", "Q9", "Q10"]     # 모델 입력 7문항 (기획서 3-10 5번)
# Lending Club 2015 값 (조사 8-1 기준, 02 노트북 4번에서 재현)
LC_POINTS = {"Q9": {"0개": 0, "1~2개": 0.41, "3개 이상": 0.88},
             "Q10": {"1년 미만": 0.49, "1~3년": 0.49, "3~5년": 0.49, "5~10년": 0.39, "10년 이상": 0}}
# NICE 신용평점별 인원 비율 % (NICE평가정보 개인신용평가 관련 통계자료, 2025년 12월, 전체 49,687,230명)
NICE_SHARE = {"BAND1": 47.9, "BAND2": 21.6, "BAND3": 24.7, "BAND4": 1.2, "BAND5": 4.6}
BAND_EDGES = [900, 800, 700, 600]       # CREDIT-01 고정 구간 (BAND1 900 초과 ~ BAND5 600 이하)
# 단조 맞추기: 점수 순서가 뒤바뀌는 이웃 보기를 묶어 같은 점수로 학습 (화면 보기는 그대로, 2026-11-13 결정)
# Q2: 문항 1과 같은 사건이라 30일 이상 위험이 문항 1로 몰림 / Q10: Lending Club 근거가 5년 미만 한 값
MERGE = {"Q2": {"10일 미만": "연체 있음", "10~29일": "연체 있음", "30일 이상": "연체 있음"},
         "Q10": {"1년 미만": "5년 미만", "1~3년": "5년 미만", "3~5년": "5년 미만"}}
# 상한을 거는 답 (연체, 채무조정, 기획서 3-10 7번). 최대 점수는 4번에서 데이터로 정함, 실제로 고른 답에만 적용
CAP_ANSWERS = [("Q1", ["1회"]), ("Q1", ["2회 이상"]), ("Q2", ["30일 이상"]), ("Q3", ["있음"])]
BAND_TOP = {"BAND2": 900, "BAND3": 800, "BAND4": 700, "BAND5": 600}     # 그 구간의 최고 점수

## 0. 읽기, 보기 나누기

In [ ]:
academy = pd.read_csv(CSV_DIR / "financial_credit_synthetic_155746x207.csv", usecols=SOURCE_COLUMNS + [LABEL])
y_original = (academy[LABEL] > 0).astype(int)
answers = to_answers(academy)
TARGET_RATE = y_original.mean()
print("행:", len(academy), "/ 원래 정답 연체율: %.2f%%" % (TARGET_RATE * 100))


def group_of(q, option, merge):
    """보기 → 묶은 보기 이름 (묶지 않은 보기는 그대로)"""
    return merge.get(q, {}).get(option, option)


def onehot(questions, merge=None):
    """보기 → 0/1 칸 (첫 보기 = 기준이라 칸 없음, merge가 있으면 묶은 보기로)"""
    merge = merge or {}
    columns = {}
    for q in questions:
        groups = list(dict.fromkeys(group_of(q, o, merge) for o in OPTIONS[q]))     # 화면 순서 유지
        labels = answers[q].astype(str).map(lambda o: group_of(q, o, merge))
        columns[q] = pd.Categorical(labels, categories=groups)
    return pd.get_dummies(pd.DataFrame(columns, index=answers.index), drop_first=True).astype(float)

## 1. 정답 다시 만들기 (기획서 3-9 정답식, 3-10 3번)
- 문항 1, 2, 3, 7, 8: 학원 원래 정답으로 맞춘 로지스틱의 로그 오즈
- 문항 9, 10: Lending Club 값 / 문항 4, 5, 6: 0
- 잡음 표준편차 0.5를 더하고, 절편만 바꿔 연체율 기대값을 원래 정답과 같게 맞춘 뒤 정답을 뽑음

In [ ]:
rng = np.random.default_rng(SEED)

X_signal = onehot(SIGNAL_QUESTIONS)
signal_model = LogisticRegression(max_iter=1000).fit(X_signal, y_original)
signal = X_signal.values @ signal_model.coef_[0]

lc = sum(answers[q].map(points).astype(float).values for q, points in LC_POINTS.items())
logit = signal + lc + rng.normal(0, NOISE_SD, len(academy))

# 절편 찾기 (나눠 좁히기): 기대 연체율 = 원래 정답 연체율
low, high = -15.0, 5.0
for _ in range(60):
    intercept = (low + high) / 2
    expected = (1 / (1 + np.exp(-(logit + intercept)))).mean()
    low, high = (intercept, high) if expected < TARGET_RATE else (low, intercept)
prob = 1 / (1 + np.exp(-(logit + intercept)))
y = (rng.random(len(academy)) < prob).astype(int)
print("절편 %.3f / 기대 연체율 %.2f%% / 뽑은 정답 연체율 %.2f%%" % (intercept, prob.mean() * 100, y.mean() * 100))

## 2. 학습 (기획서 3-10 4~5번)
- 학습 8 : 평가 2 (정답 비율 유지), 입력은 점수가 있는 7문항
- 문항 2, 10은 묶은 보기로 학습 (`MERGE`, 단조 맞추기). 묶기 전에는 문항 2 '30일 이상'이 '10일 미만'보다 덜 깎이는 순서 뒤바뀜이 있었음 (AUC 0.662 → 0.661)

In [ ]:
X = onehot(SCORE_QUESTIONS, MERGE)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
model = LogisticRegression(max_iter=1000).fit(X_train, y_train)

auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
print("학습 %d행 / 평가 %d행 / AUC(평가) %.3f (기록만, 심은 규칙의 복원이라 실제 성능 아님)" % (len(X_train), len(X_test), auc))
coef = pd.Series(model.coef_[0], index=X.columns)
coef.round(3).to_frame("계수(로그 오즈, 첫 보기 = 0)")

## 3. 점수표 바꾸기 (기획서 3-10 6번 + 2026-11-13 구간 결정)
1. 학습 데이터 로그 오즈를 NICE 누적 비율(47.9 / 69.5 / 94.2 / 95.4%)로 나눠 경계 4개를 찾음
2. 선형 점수 = 기준점 − 배율 × 로그 오즈. 배율은 900점 경계와 600점 경계 사이가 300점이 되게, 기준점은 900점 경계가 900점이 되게
3. 보기 점수 = −배율 × 계수, 기본 점수 = 기준점 − 배율 × 절편 (선형 점수 = 기본 점수 + 보기 점수 합)
4. 경계표: 선형 점수의 경계 4개 + 학습 데이터 최저, 최고 → 600 / 700 / 800 / 900점, 1점, 1000점 (구간 안은 선형)

In [ ]:
risk_train = model.decision_function(X_train)            # 학습 데이터 로그 오즈 (클수록 위험)
cumulative = np.cumsum([NICE_SHARE[b] for b in ["BAND1", "BAND2", "BAND3", "BAND4"]]) / 100
risk_edges = np.quantile(risk_train, cumulative)         # 900, 800, 700, 600점 경계의 로그 오즈

scale = 300 / (risk_edges[3] - risk_edges[0])
offset = 900 + scale * risk_edges[0]
to_linear = lambda risk: offset - scale * risk

edge_linear = [to_linear(risk_train.max()), *to_linear(risk_edges[::-1]), to_linear(risk_train.min())]
edge_score = [1, *BAND_EDGES[::-1], 1000]
print("배율 %.2f / 기준점 %.2f" % (scale, offset))
print("경계표 (선형 → 점수):", [(round(a, 1), b) for a, b in zip(edge_linear, edge_score)])

questions = {}
for q, options in OPTIONS.items():
    points = [0.0 if q not in SCORE_QUESTIONS else
              -scale * coef.get(f"{q}_{group_of(q, option, MERGE)}", 0.0) for option in options]   # 첫 보기는 칸이 없어 0
    questions[q] = {"options": options, "points": [round(p, 2) + 0.0 for p in points],          # + 0.0: -0.0 표시 없앰
                    "unknown_points": round(middle_points(points), 2) + 0.0}
base_points = offset - scale * model.intercept_[0]

pd.DataFrame([(q, o, p) for q, v in questions.items() for o, p in zip(v["options"], v["points"])],
             columns=["문항", "보기", "보기 점수"])

## 4. 상한 정하기 (기획서 3-10 7번, 2026-11-13 결정: 데이터 기준)
- 상한 없이 점수만 쓰면 위험 답 그룹 일부가 실제 위험보다 높은 구간에 들어감
- 규칙: 그룹 연체율 **이하**인 구간 중 가장 위험한 구간까지만 허용 → 최대 점수 = 그 구간의 최고 점수
- 예: 그룹 연체율 16.1%, 구간 연체율 BAND3 13.8% / BAND4 21.5% → BAND3까지 → 최대 800점
- 다시 학습하면 구간 연체율이 바뀌므로 숫자를 고정하지 않고 여기서 계산

In [ ]:
def band_of(score):
    return np.select([score > 900, score > 800, score > 700, score > 600], ["BAND1", "BAND2", "BAND3", "BAND4"], "BAND5")


score_no_cap = np.interp(to_linear(risk_train), edge_linear, edge_score)
band_train = pd.Series(band_of(score_no_cap), index=X_train.index)
band_rate = (pd.Series(y_train, index=X_train.index).groupby(band_train).mean() * 100).reindex(NICE_SHARE).round(2)
print("상한 전 구간별 연체율 %:", band_rate.to_dict())

answers_train = answers.loc[X_train.index]
CAPS, rows = [], []
for q, options in CAP_ANSWERS:
    mask = answers_train[q].isin(options)
    group_rate = y_train[mask.values].mean() * 100
    allowed = [b for b in BAND_TOP if band_rate[b] <= group_rate]           # 그룹 연체율 이하인 구간 (BAND2~5)
    if not allowed:                                                        # 모든 구간보다 안전하면 상한 없음
        continue
    top_band = max(allowed, key=lambda b: band_rate[b])                    # 그중 가장 위험한 구간
    CAPS.append({"question": q, "options": options, "max_score": BAND_TOP[top_band]})
    rows.append({"답": f"{q} {'/'.join(options)}", "사람 수": int(mask.sum()), "그룹 연체율%": round(group_rate, 1),
                 "최대 구간": top_band, "최대 점수": BAND_TOP[top_band],
                 "상한에 걸려 내려가는 비율%": round((score_no_cap[mask.values] > BAND_TOP[top_band]).mean() * 100, 1)})
pd.DataFrame(rows)

## 5. 저장

In [ ]:
version = f"credit-{date.today():%Y%m%d}"
table = {
    "version": version,
    "trained_on": str(date.today()),
    "about": "보기 번호 = options 순서 + 1, 0 = 잘 모르겠어요 (unknown_points). 선형 점수 = base_points + 보기 점수 합 → edges로 점수 1~1000 → caps",
    "nice_source": "NICE평가정보 신용평점별 인원수 2025년 12월 (전체 49,687,230명)",
    "nice_share": NICE_SHARE,
    "base_points": round(base_points, 2),
    "questions": {q: v for q, v in questions.items()},
    "edges": {"linear": [round(v, 2) for v in edge_linear], "score": edge_score},
    "caps": CAPS,
}
SCORE_TABLE_PATH.write_text(json.dumps(table, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

log = {
    "version": version, "trained_on": str(date.today()), "seed": SEED, "noise_sd": NOISE_SD,
    "rows": {"all": len(academy), "train": len(X_train), "test": len(X_test)},
    "label_rate": {"original": round(TARGET_RATE, 4), "expected": round(float(prob.mean()), 4), "drawn": round(float(y.mean()), 4)},
    "columns": SOURCE_COLUMNS, "score_questions": SCORE_QUESTIONS, "lc_points": LC_POINTS, "merge": MERGE,
    "coef": {k: round(v, 4) for k, v in coef.items()}, "intercept": round(float(model.intercept_[0]), 4),
    "scale": round(scale, 4), "offset": round(offset, 4), "auc_test": round(auc, 4),
}
TRAIN_LOG_PATH.write_text(json.dumps(log, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print("저장:", SCORE_TABLE_PATH.name, "/", TRAIN_LOG_PATH.name)

## 6. 확인: 저장한 점수표로 다시 계산 (`score_answers`)
- 상한 전 점수가 모델 로그 오즈로 바꾼 점수와 같은지 (기획서 3-10 평가, 차이는 반올림만)
- 구간 비율이 NICE와 맞는지, 연체율이 900 초과 → 600 이하로 갈수록 높아지는지

In [ ]:
saved = load_score_table()
no_cap = {**saved, "caps": []}
codes = {int(q[1:]): answers_train[q].cat.codes.values + 1 for q in OPTIONS}    # 보기 번호 (1부터)


def scores_of(table):
    result = []
    for i in range(len(answers_train)):
        row = {no: [int(c[i])] for no, c in codes.items()}
        row[5] = []
        result.append(score_answers(table, row)[0])
    return np.array(result)


table_no_cap = scores_of(no_cap)
table_score = scores_of(saved)
print("상한 전: 점수표 계산 - 모델 계산 최대 차이 %.2f점" % np.abs(table_no_cap - score_no_cap).max())

band_final = pd.Series(band_of(table_score), index=X_train.index)
pd.DataFrame({"NICE%": NICE_SHARE,
              "상한 전%": (band_train.value_counts(normalize=True) * 100).reindex(NICE_SHARE).round(1),
              "상한 후%": (band_final.value_counts(normalize=True) * 100).reindex(NICE_SHARE).round(1),
              "상한 후 연체율%": (pd.Series(y_train, index=X_train.index).groupby(band_final).mean() * 100).reindex(NICE_SHARE).round(2)})

## 7. 학습 결과 보고서 (2026-11-13 실행 기준, SEED 42)

### 7-1. 정답 다시 만들기

| 항목 | 결과 | 근거 |
| --- | --- | --- |
| 학원 신호 (문항 1, 2, 3, 7, 8) | 원래 정답으로 맞춘 5문항 로지스틱의 로그 오즈 | 기획서 3-9 정답식, 02 보고서 6-2 (학원 신호 있음) |
| 문항 9, 10 | Lending Club 값: 문항 9 +0.41 / +0.88, 문항 10 5년 미만 +0.49 / 5~10년 +0.39 / 10년 이상 0 | 조사 8-1 기준, 02 노트북 4번 재현 |
| 문항 4, 5, 6 | 0 | 기획서 3-9 (근거 없음 또는 Lending Club과 방향 반대) |
| 잡음, 연체율 | 잡음 표준편차 0.5, 절편 −2.944로 기대 연체율 9.85% (원래 정답과 같음). 뽑은 정답은 9.73% (난수 오차) | 기획서 3-9 |

### 7-2. 학습

| 항목 | 결과 | 근거 |
| --- | --- | --- |
| 나누기 | 학습 124,596행 / 평가 31,150행 (정답 비율 유지) | 기획서 3-10 4번 |
| 모델 | 로지스틱 회귀, 입력 7문항 (1, 2, 3, 7, 8, 9, 10) | 기획서 3-10 5번 |
| 보기 묶기 | 문항 2 '10일 미만, 10~29일, 30일 이상' → '연체 있음', 문항 10 '1년 미만, 1~3년, 3~5년' → '5년 미만'. 화면 보기는 그대로, 묶인 보기는 같은 점수 | 묶기 전 문항 2 '30일 이상'이 '10일 미만'보다 덜 깎임 (문항 1과 같은 사건이라 위험이 문항 1로 몰림, '30일 이상'의 95%가 문항 1 연체 있음). 문항 10은 Lending Club 근거가 5년 미만 한 값 |
| AUC (평가) | 0.661 (묶기 전 0.662) | 기록만. 심은 정답식을 되찾은 정도라 실제 신용 성능이 아님 (기획서 3-10 평가, 조사 9-2의 0.661과 같은 수준) |

### 7-3. 점수표

- 선형 점수 = 기본 점수 + 보기 점수 합, 보기 점수 = −배율 × 계수 (배율 236.03, 기준점 324.32, 기획서 3-10 6번)
- 보기 점수 (순서가 모두 위험 순서와 같음)

| 문항 | 보기 점수 |
| --- | --- |
| 1 연체 1년 | 없음 0 / 1회 −133.1 / 2회 이상 −281.9 |
| 2 최장 연체 | 없음 0 / 연체 있음(10일 미만 ~ 30일 이상) −73.2 |
| 3 채무조정 | 없음 0 / 있음 −56.4 |
| 7 카드 사용 비율 | 카드 없음 0 / 30 미만 +74.2 / 30~60 +57.2 / 60~90 +31.6 / 90 이상 +31.4 |
| 8 현금서비스 | 없음 0 / 1~2개월 −67.8 / 3개월 이상 −174.3 |
| 9 새 대출, 카드 | 0개 0 / 1~2개 −96.2 / 3개 이상 −193.4 |
| 10 거래 기간 | 5년 미만 0 / 5~10년 +24.9 / 10년 이상 +113.9 |
| 4, 5, 6 | 0 (상태와 팁에만 씀) |

- '잘 모르겠어요': 그 문항의 가운데 보기 점수, 보기 수가 짝수면 가운데 두 보기 평균 (2026-11-13 결정)

### 7-4. 점수 구간 (경계표)

- 학습 데이터 로그 오즈를 NICE 누적 비율로 나눈 경계 4개가 정확히 600 / 700 / 800 / 900점에 오도록 맞춤, 구간 안은 선형 (2026-11-13 결정)
- 경계표 (선형 점수 → 점수): 128.3 → 1 / 600.0 → 600 / 641.2 → 700 / 853.9 → 800 / 900.0 → 900 / 1064.0 → 1000. 끝을 넘으면 1 또는 1000
- 근거: NICE평가정보 신용평점별 인원수 2025년 12월 (전체 49,687,230명). 구간 판정은 CREDIT-01 고정 구간(900 초과 ~ 600 이하)을 그대로 씀

### 7-5. 상한

- 규칙: 그룹 연체율 이하인 구간 중 가장 위험한 구간까지만 허용, 실제로 고른 답에만 적용 ('잘 모르겠어요'는 안 걸림). 기획서 3-10 7번, 2026-11-13 결정
- 상한 전 구간 연체율: BAND1 5.80 / BAND2 8.36 / BAND3 13.77 / BAND4 21.50 / BAND5 30.74%

| 답 | 사람 수 | 그룹 연체율 | 최대 점수 | 내려가는 비율 |
| --- | --- | --- | --- | --- |
| 문항 1 1회 | 16,051 | 16.1% | 800 (BAND3) | 11.8% |
| 문항 1 2회 이상 | 4,945 | 30.3% | 700 (BAND4) | 9.6% |
| 문항 2 30일 이상 | 13,537 | 20.6% | 800 (BAND3) | 6.3% |
| 문항 3 있음 | 3,498 | 22.5% | 700 (BAND4) | 49.0% |

- 문항 3은 그룹 연체율(22.5%)이 BAND4(21.5%)보다 1%p 높아 BAND4까지가 됨. 경계에 가까워 다시 학습하면 800점으로 바뀔 수 있음 (그래서 숫자를 고정하지 않고 4번에서 계산)

### 7-6. 확인 (학습 데이터, 저장한 점수표로 `score_answers` 계산)

| 구간 | NICE | 상한 전 | 상한 후 | 상한 후 연체율 |
| --- | --- | --- | --- | --- |
| BAND1 900 초과 | 47.9% | 46.9% | 46.1% | 5.77% |
| BAND2 801~900 | 21.6% | 22.0% | 20.8% | 8.34% |
| BAND3 701~800 | 24.7% | 25.0% | 25.4% | 13.28% |
| BAND4 601~700 | 1.2% | 1.4% | 3.1% | 17.57% |
| BAND5 600 이하 | 4.6% | 4.6% | 4.6% | 30.75% |

- 상한 전 점수표 계산과 모델 계산의 차이: 최대 0.51점 (보기 점수 반올림만)
- 연체율이 900 초과 → 600 이하로 갈수록 계속 높아짐 (기획서 3-10 평가 기준)

### 7-7. 한계

- 학원 데이터는 교육용 합성, Lending Club은 미국 승인 대출만이고 정답 정의도 다름 (조사 8-4, 9-3). 점수는 공식 신용점수가 아님 (CREDIT-01 고지)
- BAND1이 NICE보다 약 1~2%p 적음: 가장 좋은 답 조합을 고른 사람이 많아 같은 점수끼리는 더 나눌 수 없음
- BAND4는 사람이 1.4%로 적어 구간 연체율이 흔들릴 수 있음 → 문항 3 상한이 경계에 걸림 (7-5)
- 상한 뒤 BAND4가 3.1%로 NICE(1.2%)보다 많아짐 (상한에 걸린 사람이 내려옴)
- 평가 데이터(20%)로 하는 구간별 정답 비율, 점수표와 모델 일치 확인은 04 평가 노트북에서 함